# importing libraries

In [1]:
!pip install gensim numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 44.4 MB/s eta 0:00:00


In [12]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from gensim.models import KeyedVectors
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, f1_score
import gensim.downloader as api

# loading dataset

In [3]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# preprocessing

In [4]:
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)
df.rename(columns={"v1": "label", "v2": "text"}, inplace=True)
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

,label,text,label_enc
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [5]:
X_train, X_test, y_train, y_test = train = train_test_split(df['text'], df['label_enc'], test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = X_train.to_numpy(), X_test.to_numpy(), y_train.to_numpy(), y_test.to_numpy()

In [6]:
y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

# vocabulary stats

In [7]:
averageWordsLength = round(sum([len(i.split()) for i in df['text']]) / len(df['text']))
totalWordsLength = len(set(" ".join(df['text']).split()))
print(f"Data Loaded. Training samples: {len(X_train)}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 4457
Average words per message: 16
Approximate vocabulary size: 15686


# model development

In [8]:
cnn = tf.keras.models.Sequential()
cnn.add(tf.keras.layers.Input(shape=(1,), dtype=tf.string))

In [9]:
text_vec = tf.keras.layers.TextVectorization(
    max_tokens=totalWordsLength,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    output_sequence_length=averageWordsLength,
)
text_vec.adapt(X_train)
cnn.add(text_vec)

In [10]:
vocab_size = text_vec.get_vocabulary()
word_index = dict(zip(vocab_size, range(len(vocab_size))))

In [13]:
ft = api.load("fasttext-wiki-news-subwords-300")

[==================================================] 100.0% 958.5/958.4MB downloaded


In [ ]:
embedding_dim = 300
embedding_matrix = np.zeros((len(vocab_size), embedding_dim))

for word, i in word_index.items():
    if word in ft:
        embedding_matrix[i] = ft[word]

In [14]:
cnn.add(tf.keras.layers.Embedding(
    input_dim=len(vocab_size),
    output_dim=embedding_dim,
    input_length=averageWordsLength,
    weights=[embedding_matrix],
    trainable=False
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [15]:
cnn.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation="relu"))
cnn.add(tf.keras.layers.GlobalAveragePooling1D())
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))
cnn.add(tf.keras.layers.Dense(units=2, activation='softmax'))

In [16]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 16)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 16, 300)        │     2,552,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 12, 128)        │       192,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,761,298 (10.53 MB)

 Trainable params: 208,898 (816.01 KB)

 Non-trainable params: 2,552,400 (9.74 MB)

In [19]:
cnn.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

# model training

In [20]:
history = cnn.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8547 - loss: 0.3795 - val_accuracy: 0.9632 - val_loss: 0.1150
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9692 - loss: 0.0899 - val_accuracy: 0.9659 - val_loss: 0.1004
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9733 - loss: 0.0769 - val_accuracy: 0.9668 - val_loss: 0.0959
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9814 - loss: 0.0601 - val_accuracy: 0.9713 - val_loss: 0.0883
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.9872 - loss: 0.0428 - val_accuracy: 0.9713 - val_loss: 0.0894


# model evaluation

In [21]:
test_messages = [
    "Hey, are we meeting today?",
    "WIN cash now!!! Click the link",
    "Your OTP is 348921",
    "You have won $1000!",
    "Congratulations! You have won a free lottery ticket",
    "Don't forget to submit the assignment",
    "URGENT! You won a cash prize. Call immediately!",
    "My friend won a prize yesterday"
]

# Convert the list of messages to a TensorFlow constant of strings
test_messages_tf = tf.constant(test_messages, dtype=tf.string)

preds = cnn.predict(test_messages_tf)

for msg, p in zip(test_messages, preds):
    label = "Spam" if np.argmax(p)==1 else "Ham"
    print(f"{label} → {msg} ({p[0]:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
Ham → Hey, are we meeting today? (1.00)
Ham → WIN cash now!!! Click the link (0.84)
Ham → Your OTP is 348921 (0.98)
Ham → You have won $1000! (0.69)
Spam → Congratulations! You have won a free lottery ticket (0.13)
Ham → Don't forget to submit the assignment (1.00)
Spam → URGENT! You won a cash prize. Call immediately! (0.02)
Spam → My friend won a prize yesterday (0.50)


In [22]:
print(classification_report(y_test.argmax(axis=1), cnn.predict(X_test).argmax(axis=1)))
print(f1_score(y_test.argmax(axis=1), cnn.predict(X_test).argmax(axis=1)))

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       966
           1       0.94      0.84      0.89       149

    accuracy                           0.97      1115
   macro avg       0.96      0.92      0.94      1115
weighted avg       0.97      0.97      0.97      1115

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
0.8865248226950354
